# FinChart-R2 — Phase 2C Multimodal DPO-386 (Kaggle → Hugging Face)

This Kaggle notebook continues from `Kxck/Finance_500_v1`, trains an offline multimodal DPO adapter on ChartQA **train-only** preference pairs, and creates or updates a Hugging Face model repository.

Security: create a Kaggle Secret named `HF_TOKEN` with a Hugging Face **write** token. The token is read at runtime, never embedded or printed.

The 386-pair file is currently provisional. Keep the output repository name explicitly marked `provisional` until manual audit and response-format balancing are complete. The frozen ChartQA `val[0:500]` evaluator is not used for training.

## 1. Kaggle setup

Enable a GPU and Internet in Kaggle Notebook Settings. Run the install cell in a fresh session. If Kaggle requests a restart, restart once and continue from Section 2.

In [ ]:
%pip uninstall -y unsloth unsloth_zoo torchao
%pip install -q -U --no-cache-dir "trl[peft]==0.29.0" "transformers>=5.0.0,<6.0.0" "datasets>=4.4.0,<6.0.0" accelerate bitsandbytes huggingface_hub qwen-vl-utils
print('Installed without replacing Kaggle Pillow; continue directly to Section 2.')

## 2. Runtime and Hugging Face authentication

In [ ]:
import importlib.util
import json
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'  # avoid DataParallel replication of 4-bit Params4bit
import platform
import random
import warnings
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

if importlib.util.find_spec('torchao') is not None:
    raise RuntimeError(
        'TorchAO is still installed. Rerun Section 1, restart the Kaggle session, '
        'and continue from Section 2. This notebook uses bitsandbytes, not TorchAO.'
    )

import datasets
import peft
import torch
import transformers
import trl
from huggingface_hub import HfApi, login
from kaggle_secrets import UserSecretsClient

if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator before continuing.')
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        f'Expected exactly one visible GPU, found {torch.cuda.device_count()}. '
        'Restart the Kaggle session and begin again from Section 2.'
    )
if trl.__version__ != '0.29.0':
    raise RuntimeError(
        f'Expected TRL 0.29.0, found {trl.__version__}. '
        'Rerun Section 1, restart the session, and start again from Section 2.'
    )
try:
    from trl.trainer.dpo_trainer import DataCollatorForVisionPreference
except ImportError as error:
    raise RuntimeError(
        f'TRL {trl.__version__} has no native vision preference collator. '
        'Rerun Section 1 and restart the session.'
    ) from error

try:
    HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as error:
    raise RuntimeError(
        'Add a Kaggle Secret named HF_TOKEN with Hugging Face write permission.'
    ) from error
if not HF_TOKEN:
    raise RuntimeError('HF_TOKEN is empty.')

login(token=HF_TOKEN, add_to_git_credential=False)
hf_api = HfApi(token=HF_TOKEN)
hf_identity = hf_api.whoami()
HF_USERNAME = hf_identity.get('name') or hf_identity.get('fullname')
print('Authenticated Hugging Face account:', HF_USERNAME)

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
gpu = torch.cuda.get_device_properties(0)
BF16 = torch.cuda.is_bf16_supported()
print({
    'python': platform.python_version(),
    'torch': torch.__version__,
    'transformers': transformers.__version__,
    'trl': trl.__version__,
    'peft': peft.__version__,
    'datasets': datasets.__version__,
    'gpu': gpu.name,
    'gpu_memory_GB': round(gpu.total_memory / 2**30, 1),
    'bf16': BF16,
})

## 3. Configuration and fixed Kaggle input path

This run uses the fixed Kaggle input `/kaggle/input/datasets/khangkxcp/dpo300/phase2c_teacher_v1_dpo_candidates_provisional.jsonl` and publishes to `Kxck/Finance_500_v1_DPO_386_provisional`.

In [ ]:
BASE_MODEL = 'Qwen/Qwen3-VL-4B-Instruct'
SFT_ADAPTER_ID = 'Kxck/Finance_500_v1'
DATASET_NAME = 'HuggingFaceM4/ChartQA'
PAIR_JSONL = Path('/kaggle/input/datasets/khangkxcp/dpo300/phase2c_teacher_v1_dpo_candidates_provisional.jsonl')
EXPECTED_PAIRS = 386
PAIR_EVAL_FRACTION = 0.10
ALLOW_PROVISIONAL_PAIRS = True
ALLOW_SCHEMA_MISMATCH = True
RUN_TRAINING = True
PUSH_TO_HUB = True
HF_PRIVATE_REPO = False
HUB_MODEL_ID = 'Kxck/Finance_500_v1_DPO_386_provisional'

WORK_DIR = Path('/kaggle/working/finchart_r2_phase2c_dpo_386_unsloth')
CHECKPOINT_DIR = WORK_DIR / 'checkpoints'
ADAPTER_DIR = WORK_DIR / 'adapter_sft_dpo_386'
for directory in (WORK_DIR, CHECKPOINT_DIR, ADAPTER_DIR):
    directory.mkdir(parents=True, exist_ok=True)

if not PAIR_JSONL.is_file():
    raise FileNotFoundError(
        f'DPO pair file not found: {PAIR_JSONL}. Attach the khangkxcp/dpo300 Kaggle Dataset.'
    )
if HUB_MODEL_ID.split('/', 1)[0] != HF_USERNAME:
    warnings.warn(
        f'Authenticated as {HF_USERNAME}, but HUB_MODEL_ID targets {HUB_MODEL_ID}. '
        'This is valid only if the token can write to that namespace.'
    )
print({'pair_file': str(PAIR_JSONL), 'work_dir': str(WORK_DIR), 'hub_model_id': HUB_MODEL_ID})

## 4. Validate train-only preference pairs

In [ ]:
import hashlib
import re

records = [
    json.loads(line)
    for line in PAIR_JSONL.read_text(encoding='utf-8').splitlines()
    if line.strip()
]
required = {
    'dataset_index', 'image_split', 'image_index', 'prompt',
    'chosen', 'rejected', 'ground_truth', 'manual_audit_status'
}
for line_number, row in enumerate(records, 1):
    missing = required - set(row)
    if missing:
        raise ValueError(f'Line {line_number} is missing {sorted(missing)}')
    if row['image_split'] != 'train':
        raise RuntimeError(f'Validation leakage at line {line_number}: {row["image_split"]}')
    if not all(str(row[key]).strip() for key in ('prompt', 'chosen', 'rejected')):
        raise ValueError(f'Empty preference field at line {line_number}')
    if str(row['chosen']).strip() == str(row['rejected']).strip():
        raise ValueError(f'Identical chosen/rejected at line {line_number}')

if len(records) != EXPECTED_PAIRS:
    raise ValueError(f'Expected {EXPECTED_PAIRS} pairs, found {len(records)}')
indices = [int(row['dataset_index']) for row in records]
image_refs = {(row['image_split'], int(row['image_index'])) for row in records}
if len(set(indices)) != len(records) or len(image_refs) != len(records):
    raise ValueError('Duplicate dataset indices or image references detected.')

approved = {'APPROVED', 'VALIDATED', 'AUDITED_APPROVED'}
audit_statuses = Counter(str(row['manual_audit_status']) for row in records)
pending_count = sum(n for status, n in audit_statuses.items() if status not in approved)
if pending_count and not ALLOW_PROVISIONAL_PAIRS:
    raise RuntimeError(f'{pending_count} pairs are not manually approved.')

fields = ('Relevant values', 'Operation', 'Calculation', 'Answer')
def signature(text):
    return tuple(
        field for field in fields
        if re.search(rf'^\s*{re.escape(field)}\s*:', str(text), re.I | re.M)
    )
schema_matched = sum(signature(r['chosen']) == signature(r['rejected']) for r in records)
if schema_matched < len(records) and not ALLOW_SCHEMA_MISMATCH:
    raise RuntimeError(f'Only {schema_matched}/{len(records)} pairs have matched response schemas.')
if pending_count:
    warnings.warn('PROVISIONAL RUN: manual audit is incomplete.')
if schema_matched < len(records):
    warnings.warn(f'FORMAT-BIAS RISK: {schema_matched}/{len(records)} schemas match.')
pair_sha256 = hashlib.sha256(PAIR_JSONL.read_bytes()).hexdigest()
print({
    'pairs': len(records),
    'unique_images': len(image_refs),
    'split': sorted({r['image_split'] for r in records}),
    'audit_statuses': dict(audit_statuses),
    'same_field_signature': schema_matched,
    'sha256': pair_sha256,
})

## 5. Rehydrate ChartQA images and build the vision preference dataset

In [ ]:
from datasets import Dataset, load_dataset

chartqa_train = load_dataset(DATASET_NAME, split='train', token=HF_TOKEN)
examples = []
for row in records:
    image_index = int(row['image_index'])
    source = chartqa_train[image_index]
    source_question = str(source.get('query', source.get('question', ''))).strip()
    if source_question != str(row['prompt']).strip():
        raise ValueError(f'Question mismatch at dataset_index={row["dataset_index"]}')
    examples.append({
        'image': source['image'].convert('RGB'),
        'prompt': [{'role': 'user', 'content': str(row['prompt'])}],
        'chosen': [{'role': 'assistant', 'content': str(row['chosen'])}],
        'rejected': [{'role': 'assistant', 'content': str(row['rejected'])}],
    })

preference_dataset = Dataset.from_list(examples)
splits = preference_dataset.train_test_split(
    test_size=PAIR_EVAL_FRACTION, seed=SEED, shuffle=True
)
train_dataset = splits['train']
eval_dataset = splits['test']
print(preference_dataset)
print({'train_pairs': len(train_dataset), 'eval_pairs': len(eval_dataset)})

## 6. Quantize the official Qwen3-VL base and load the SFT-408 adapter

In [ ]:
from peft import PeftModel, prepare_model_for_kbit_training
from transformers import AutoModelForImageTextToText, AutoProcessor, BitsAndBytesConfig

compute_dtype = torch.bfloat16 if BF16 else torch.float16
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)
processor = AutoProcessor.from_pretrained(BASE_MODEL, token=HF_TOKEN)
processor.tokenizer.padding_side = 'left'
if processor.tokenizer.pad_token_id is None:
    processor.tokenizer.pad_token = processor.tokenizer.eos_token
base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL,
    device_map={'': 0},
    dtype=compute_dtype,
    attn_implementation='sdpa',
    quantization_config=quantization_config,
    token=HF_TOKEN,
)
base_model = prepare_model_for_kbit_training(
    base_model, use_gradient_checkpointing=True
)
model = PeftModel.from_pretrained(
    base_model, SFT_ADAPTER_ID, is_trainable=True, token=HF_TOKEN
)
model.config.use_cache = False
model.enable_input_require_grads()
model.print_trainable_parameters()
print('Loaded official Qwen3-VL with on-the-fly NF4 + trainable SFT-408 adapter.')

## 7. Configure TRL multimodal DPO

TRL 0.29 supplies `DataCollatorForVisionPreference`. The official Qwen3-VL base is quantized on-the-fly with bitsandbytes NF4 before attaching the existing SFT adapter. This avoids stale or lost FP4 state in pre-quantized vision DeepStack layers.

In [ ]:
from trl import DPOConfig
import importlib
import trl.trainer.dpo_trainer as dpo_module

# TRL 0.29.0 omits Qwen3-VL's mm_token_type_ids from both model-forward
# allowlists. Patch only those two exact allowlists, then reload the module.
trainer_source_path = Path(dpo_module.__file__).resolve()
trainer_source = trainer_source_path.read_text(encoding='utf-8')
trl_029_forward_keys = (
    'for key in ("pixel_values", "pixel_attention_mask", "image_grid_thw", '
    '"image_sizes", "token_type_ids"):'
)
qwen3vl_forward_keys = (
    'for key in ("pixel_values", "pixel_attention_mask", "image_grid_thw", '
    '"image_sizes", "token_type_ids", "mm_token_type_ids"):'
)
if qwen3vl_forward_keys not in trainer_source:
    occurrence_count = trainer_source.count(trl_029_forward_keys)
    if occurrence_count != 2:
        raise RuntimeError(
            'Refusing to patch unexpected TRL source: expected two model-forward allowlists, '
            f'found {occurrence_count} in {trainer_source_path}.'
        )
    trainer_source_path.write_text(
        trainer_source.replace(trl_029_forward_keys, qwen3vl_forward_keys),
        encoding='utf-8',
    )
    importlib.invalidate_caches()
    dpo_module = importlib.reload(dpo_module)
elif trl_029_forward_keys in trainer_source:
    raise RuntimeError('TRL source is only partially patched; start a fresh Kaggle session.')

DPOTrainer = dpo_module.DPOTrainer
DataCollatorForVisionPreference = dpo_module.DataCollatorForVisionPreference
patched_source = trainer_source_path.read_text(encoding='utf-8')
if patched_source.count(qwen3vl_forward_keys) != 2:
    raise RuntimeError('Failed to enable mm_token_type_ids in both TRL DPO forwards.')
print('Patched TRL 0.29 policy + reference forwards for Qwen3-VL mm_token_type_ids.')

class Qwen3VLVisionPreferenceCollator:
    """Repair TRL 0.29 mm_token_type_ids for prompt + completion batches."""

    def __init__(self, processor):
        self.base_collator = DataCollatorForVisionPreference(processor=processor)

    def __call__(self, examples):
        if len(examples) != 1:
            raise RuntimeError(
                'This Qwen3-VL collator repair requires per-device batch size 1.'
            )
        batch = self.base_collator(examples)
        if 'mm_token_type_ids' not in batch:
            raise RuntimeError('Qwen3-VL processor did not return mm_token_type_ids.')
        mm_ids = batch['mm_token_type_ids']
        input_ids = batch['input_ids']
        if mm_ids.shape[0] != input_ids.shape[0]:
            raise RuntimeError(
                'mm_token_type_ids and input_ids have different batch dimensions.'
            )
        missing_tokens = input_ids.shape[1] - mm_ids.shape[1]
        if missing_tokens < 0:
            raise RuntimeError('mm_token_type_ids is longer than input_ids.')
        if missing_tokens:
            completion_types = torch.zeros(
                (mm_ids.shape[0], missing_tokens),
                dtype=mm_ids.dtype,
                device=mm_ids.device,
            )
            batch['mm_token_type_ids'] = torch.cat(
                (mm_ids, completion_types), dim=1
            )
        if batch['mm_token_type_ids'].shape != input_ids.shape:
            raise RuntimeError('Failed to align Qwen3-VL mm_token_type_ids.')
        return batch

qwen3vl_dpo_collator = Qwen3VLVisionPreferenceCollator(processor)

dpo_args = DPOConfig(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    num_train_epochs=1,
    learning_rate=2e-6,
    warmup_steps=3,
    beta=0.1,
    loss_type=['sigmoid'],
    max_length=None,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
    optim='paged_adamw_8bit',
    max_grad_norm=1.0,
    bf16=BF16,
    fp16=not BF16,
    tf32=gpu.major >= 8,
    logging_steps=1,
    logging_first_step=True,
    eval_strategy='epoch',
    save_strategy='epoch',
    save_total_limit=2,
    remove_unused_columns=False,
    dataloader_num_workers=0,
    report_to='none',
    seed=SEED,
)
trainer = DPOTrainer(
    model=model,
    ref_model=None,
    args=dpo_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=processor,
    data_collator=qwen3vl_dpo_collator,
)
print('DPO collator:', type(trainer.data_collator).__name__)

## 8. Mandatory multimodal preflight

Do not train if this cell fails. It prevents accidental text-only DPO.

In [ ]:
import copy
import gc

collator_name = type(trainer.data_collator).__name__
if 'VisionPreference' not in collator_name:
    raise RuntimeError(
        f'STOPPED: expected TRL DataCollatorForVisionPreference, got {collator_name}.'
    )
batch = next(iter(trainer.get_train_dataloader()))
required_keys = {
    'input_ids', 'attention_mask', 'completion_mask', 'pixel_values',
    'mm_token_type_ids',
}
missing = required_keys - set(batch)
if missing:
    raise RuntimeError(f'STOPPED: multimodal DPO batch is missing {sorted(missing)}.')
sequence_rows = int(batch['input_ids'].shape[0])
if sequence_rows < 2 or sequence_rows % 2 != 0:
    raise RuntimeError(
        f'Expected an even number of chosen/rejected sequence rows, got {sequence_rows}.'
    )
if batch['attention_mask'].shape[0] != sequence_rows:
    raise RuntimeError('attention_mask and input_ids have inconsistent batch dimensions.')
if batch['completion_mask'].shape[0] != sequence_rows:
    raise RuntimeError('completion_mask and input_ids have inconsistent batch dimensions.')
if batch['mm_token_type_ids'].shape != batch['input_ids'].shape:
    raise RuntimeError(
        'mm_token_type_ids must match the complete prompt + completion input shape.'
    )
pair_rows = sequence_rows // 2
completion_tokens = batch['completion_mask'].sum(dim=1)
if (completion_tokens <= 0).any():
    raise RuntimeError('At least one chosen/rejected sequence has no supervised completion tokens.')
distinct_pairs = sum(
    not torch.equal(batch['input_ids'][i], batch['input_ids'][i + pair_rows])
    for i in range(pair_rows)
)
if distinct_pairs != pair_rows:
    raise RuntimeError(
        f'Only {distinct_pairs}/{pair_rows} chosen/rejected pairs differ after collation.'
    )
forward_keys = {
    'input_ids', 'attention_mask', 'pixel_values', 'image_grid_thw',
    'pixel_values_videos', 'video_grid_thw', 'mm_token_type_ids',
}
smoke_inputs = {
    key: value.to(model.device) if hasattr(value, 'to') else value
    for key, value in batch.items()
    if key in forward_keys
}
model.eval()
with torch.inference_mode():
    smoke_output = model(**smoke_inputs, use_cache=False, logits_to_keep=1)
if smoke_output.logits.shape[0] != sequence_rows:
    raise RuntimeError('Vision forward smoke test returned an unexpected batch dimension.')
prepared_batch = trainer._prepare_inputs(batch)
metrics_snapshot = copy.deepcopy(trainer._metrics)
with torch.no_grad():
    dpo_smoke_loss = trainer.compute_loss(model, prepared_batch)
if not torch.isfinite(dpo_smoke_loss).item():
    raise RuntimeError(f'DPO loss preflight is non-finite: {dpo_smoke_loss.item()}')
trainer._metrics = metrics_snapshot  # preserve required train/eval buckets and discard smoke metrics
model.train()
print('VLM DPO preflight passed.')
print('Qwen3-VL 4-bit vision forward smoke test passed.')
print('Policy + reference DPO loss smoke test passed:', float(dpo_smoke_loss))
print({'preference_pairs_in_batch': pair_rows, 'sequence_rows': sequence_rows})
print({key: tuple(value.shape) for key, value in batch.items() if hasattr(value, 'shape')})
del batch, smoke_inputs, smoke_output, prepared_batch, dpo_smoke_loss, metrics_snapshot
gc.collect()
torch.cuda.empty_cache()

## 9. Train and save the adapter

In [ ]:
from transformers.trainer_utils import get_last_checkpoint

if not RUN_TRAINING:
    raise RuntimeError('RUN_TRAINING=False; preflight passed but training is disabled.')
last_checkpoint = get_last_checkpoint(str(CHECKPOINT_DIR))
print('Resume checkpoint:', last_checkpoint)
train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
trainer.save_model(str(ADAPTER_DIR))
processor.save_pretrained(str(ADAPTER_DIR))
trainer.save_state()
print(train_result.metrics)
print('Saved adapter:', ADAPTER_DIR)

## 10. Create or update the Hugging Face model

`create_repo(..., exist_ok=True)` creates the repository on the first run. `upload_folder` creates a new commit on later runs. The secret token is not saved in the model files.

In [ ]:
def revision(repo_id):
    try:
        return hf_api.model_info(repo_id).sha
    except Exception as error:
        warnings.warn(f'Could not resolve revision for {repo_id}: {error}')
        return None

manifest = {
    'experiment': 'FinChart Phase 2C multimodal DPO-386 Kaggle pilot',
    'created_utc': datetime.now(timezone.utc).isoformat(),
    'base_model': BASE_MODEL,
    'quantization_source': 'on-the-fly bitsandbytes NF4',
    'source_sft_adapter': SFT_ADAPTER_ID,
    'base_model_revision': revision(BASE_MODEL),
    'source_sft_adapter_revision': revision(SFT_ADAPTER_ID),
    'pair_source_filename': PAIR_JSONL.name,
    'pair_sha256': pair_sha256,
    'pairs_total': len(records),
    'pairs_train': len(train_dataset),
    'pairs_eval': len(eval_dataset),
    'manual_audit_status': dict(audit_statuses),
    'allow_provisional_pairs': ALLOW_PROVISIONAL_PAIRS,
    'same_field_signature': schema_matched,
    'algorithm': 'offline multimodal DPO',
    'model_backend': 'Transformers/PEFT with on-the-fly bitsandbytes NF4',
    'preference_collator': type(trainer.data_collator).__name__,
    'loss_type': 'sigmoid',
    'beta': 0.1,
    'learning_rate': 2e-6,
    'epochs': 1,
    'effective_batch_size': 8,
    'train_metrics': train_result.metrics,
    'library_versions': {
        'torch': torch.__version__,
        'transformers': transformers.__version__,
        'trl': trl.__version__,
        'peft': peft.__version__,
        'datasets': datasets.__version__,
    },
    'phase1_validation_used_for_training': False,
    'reportable_model': pending_count == 0 and schema_matched == len(records),
    'next_step': 'Run the exact frozen ChartQA val[0:500] evaluator against SFT-408.',
}
manifest_path = ADAPTER_DIR / 'dpo_386_training_manifest.json'
manifest_path.write_text(json.dumps(manifest, indent=2) + '\n', encoding='utf-8')

model_card = f'''---
base_model: {BASE_MODEL}
library_name: peft
pipeline_tag: image-text-to-text
tags:
- finchart
- qwen3-vl
- dpo
- unsloth
---

# FinChart-R2 SFT-408 + DPO-386 provisional adapter

This adapter continues from `{SFT_ADAPTER_ID}` using train-only ChartQA multimodal preference pairs.

Status: **provisional diagnostic**. Manual audit and frozen ChartQA val[0:500] evaluation are still required.
'''
(ADAPTER_DIR / 'README.md').write_text(model_card, encoding='utf-8')

if PUSH_TO_HUB:
    hf_api.create_repo(
        repo_id=HUB_MODEL_ID,
        repo_type='model',
        private=HF_PRIVATE_REPO,
        exist_ok=True,
    )
    commit = hf_api.upload_folder(
        repo_id=HUB_MODEL_ID,
        repo_type='model',
        folder_path=str(ADAPTER_DIR),
        path_in_repo='',
        commit_message='Update FinChart-R2 SFT-408 + DPO-386 provisional adapter',
    )
    print('Created/updated:', f'https://huggingface.co/{HUB_MODEL_ID}')
    print('Commit:', getattr(commit, 'oid', getattr(commit, 'commit_url', 'uploaded')))
else:
    print('PUSH_TO_HUB=False; adapter remains in /kaggle/working.')

## 11. Evaluation handoff

Training loss and preference reward accuracy do not establish ChartQA improvement. Load the published adapter in the exact frozen Phase 1 evaluator and compare it with SFT-408 (`345/500`, 69.0%) on the same `val[0:500]` rows. Report DPO fixes, regressions, both-correct, both-wrong, and failure-category changes.